# 04 — Modelo Baseline (Classical ML)

Comparamos 3 configuraciones para predecir la dirección del precio en Polymarket:

| # | Modelo | Features |
|---|--------|----------|
| 0 | Majority class baseline | — |
| 1 | Random Forest | market features (8 vars) |
| 2 | Gradient Boosting | market features (8 vars) |

Métricas: **Accuracy, Precision, Recall, F1-score** (macro y weighted).  
Calibración de probabilidades: **Isotonic Regression**.  
Split: temporal 70/30 (sin shuffle).

In [1]:
import sys
import pickle
from pathlib import Path

ROOT = Path("__file__").resolve().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PROC_DIR = ROOT / "data" / "processed"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    brier_score_loss,
)   

from src.features import temporal_split

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
RANDOM_SEED = 42

ImportError: cannot import name 'calibration_curve' from 'sklearn.metrics' (c:\Users\oscar\repos\predikt\.venv\Lib\site-packages\sklearn\metrics\__init__.py)

## 1. Cargar dataset procesado

In [ ]:
data = pd.read_csv(PROC_DIR / "features_all.csv", parse_dates=["date"])
print(f"Shape: {data.shape}")
print(f"Columnas: {list(data.columns)}")
print(f"\nBalance de clases:\n{data['label'].value_counts()}")
data.head()

## 2. Split temporal 70/30

In [ ]:
FEATURE_COLS = ["ret_1d", "ret_3d", "ret_7d", "ma7", "ma14", "ma_ratio", "vol7", "price"]

train, test = temporal_split(data, train_ratio=0.70)

X_train = train[FEATURE_COLS].values
X_test  = test[FEATURE_COLS].values
y_train = train["label"].values
y_test  = test["label"].values

print(f"Train: {len(train)} filas | {train['date'].min().date()} → {train['date'].max().date()}")
print(f"Test : {len(test)}  filas | {test['date'].min().date()} → {test['date'].max().date()}")
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

## 3. Entrenamiento de modelos

In [ ]:
def evaluate(name, y_true, y_pred, y_prob=None):
    result = {
        "Modelo"            : name,
        "Accuracy"          : round(accuracy_score(y_true, y_pred), 4),
        "Precision (macro)" : round(precision_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "Recall (macro)"    : round(recall_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "F1 (macro)"        : round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "F1 (weighted)"     : round(f1_score(y_true, y_pred, average="weighted", zero_division=0), 4),
    }
    if y_prob is not None:
        result["Brier Score"] = round(brier_score_loss(y_true, y_prob), 4)
    return result

results = []
trained_models = {}

In [ ]:
# Majority Class Baseline
majority_class = int(np.bincount(y_train).argmax())
y_pred_baseline = np.full_like(y_test, fill_value=majority_class)
results.append(evaluate("0 — Majority Baseline", y_test, y_pred_baseline))
print(f"Baseline: siempre predice clase {majority_class}")

In [ ]:
# Modelo 1: Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=RANDOM_SEED)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_prob_rf  = rf.predict_proba(X_test)[:, 1]
results.append(evaluate("1 — Random Forest", y_test, y_pred_rf, y_prob_rf))
trained_models["RandomForest"] = rf
print("Modelo 1 entrenado.")

In [ ]:
# Modelo 2: Gradient Boosting
gb = GradientBoostingClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.8, random_state=RANDOM_SEED
)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
y_prob_gb  = gb.predict_proba(X_test)[:, 1]
results.append(evaluate("2 — Gradient Boosting", y_test, y_pred_gb, y_prob_gb))
trained_models["GradientBoosting"] = gb
print("Modelo 2 entrenado.")

## 4. Tabla comparativa de métricas

In [ ]:
results_df = pd.DataFrame(results).set_index("Modelo")

display(results_df.style
    .highlight_max(axis=0, color="lightgreen")
    .highlight_min(
        axis=0,
        subset=["Brier Score"] if "Brier Score" in results_df.columns else [],
        color="lightgreen"
    )
    .format("{:.4f}")
)

results_df.to_csv(PROC_DIR / "model_comparison.csv")
print("Tabla guardada en data/processed/model_comparison.csv")

## 5. Matrices de confusión

In [ ]:
pairs = [
    ("Random Forest",     y_test, y_pred_rf),
    ("Gradient Boosting", y_test, y_pred_gb),
]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), squeeze=False)
axes_flat = axes.flatten()

for i, (name, y_true, y_pred) in enumerate(pairs):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["DOWN (0)", "UP (1)"])
    disp.plot(ax=axes_flat[i], colorbar=False, cmap="Blues")
    axes_flat[i].set_title(name, fontsize=11)

plt.suptitle("Matrices de confusión — conjunto de prueba", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(PROC_DIR / "confusion_matrices.png", bbox_inches="tight")
plt.show()

## 6. Importancia de features (Random Forest)

In [ ]:
feat_imp = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

plt.figure(figsize=(9, 4))
sns.barplot(x=feat_imp.values, y=feat_imp.index, orient="h", palette="viridis")
plt.xlabel("Importancia (Gini)")
plt.title("Importancia de features — Random Forest")
plt.tight_layout()
plt.savefig(PROC_DIR / "feature_importance_rf.png", bbox_inches="tight")
plt.show()

## 7. Calibración de probabilidades (Isotonic Regression)

La regresión isotónica ajusta las probabilidades de salida del modelo para que se alineen con las frecuencias reales observadas en los datos de validación.

In [ ]:
# Fit isotonic calibration on first half of test set, evaluate on second half
mid = len(y_test) // 2
iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(y_prob_rf[:mid], y_test[:mid])
probs_calibrated = iso.predict(y_prob_rf[mid:])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (label, probs) in zip(axes, [
    ("Sin calibrar",             y_prob_rf[mid:]),
    ("Con Isotonic Regression",  probs_calibrated),
]):
    frac_pos, mean_pred = calibration_curve(y_test[mid:], probs, n_bins=8, strategy="uniform")
    ax.plot(mean_pred, frac_pos, marker="o", label=label)
    ax.plot([0, 1], [0, 1], "--", color="gray", label="Perfectamente calibrado")
    ax.set_xlabel("Probabilidad predicha")
    ax.set_ylabel("Fracción positivos reales")
    ax.set_title(f"Curva de calibración — {label}")
    ax.legend()

plt.tight_layout()
plt.savefig(PROC_DIR / "calibration_curves.png", bbox_inches="tight")
plt.show()

brier_before = brier_score_loss(y_test[mid:], y_prob_rf[mid:])
brier_after  = brier_score_loss(y_test[mid:], probs_calibrated)
print(f"Brier score sin calibrar : {brier_before:.4f}")
print(f"Brier score calibrado    : {brier_after:.4f}")

## 8. Reporte del modelo seleccionado

In [ ]:
print("=" * 55)
print("MODELO SELECCIONADO: Random Forest (market features)")
print("=" * 55)
print(classification_report(y_test, y_pred_rf, target_names=["DOWN (0)", "UP (1)"]))

## 9. Guardar modelos entrenados

In [ ]:
for name, model in trained_models.items():
    path = PROC_DIR / f"model_{name}.pkl"
    with open(path, "wb") as fh:
        pickle.dump(model, fh)
    print(f"Guardado: {path.name}")

with open(PROC_DIR / "isotonic_calibrator.pkl", "wb") as fh:
    pickle.dump(iso, fh)
print("Guardado: isotonic_calibrator.pkl")